# Helpstral Inference Server

Loads the fine-tuned Helpstral LoRA adapter ([BenBarr/helpstral](https://huggingface.co/BenBarr/helpstral)) and serves it as an API via ngrok.

**Steps:**
1. Run all cells
2. Copy the ngrok URL printed at the bottom
3. Set `HELPSTRAL_ENDPOINT=<ngrok_url>` in your `.env` file
4. Your server will now use your fine-tuned model for Helpstral inference

**Runtime:** Use a T4 GPU or higher (Pixtral 12B + LoRA needs ~12–16 GB VRAM)

In [1]:
!pip install -q transformers peft accelerate flask pyngrok huggingface_hub torch pillow bitsandbytes

## Load model

In [2]:
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel

# Base: mistral-community/pixtral-12b — the standard HuggingFace port of Pixtral 12B.
# The adapter was trained on unsloth/pixtral-12b-2409-bnb-4bit. Unsloth's 4-bit model
# uses identical architecture and layer names — it only differs in pre-quantisation.
# Loading the LoRA adapter onto the community port with fresh 4-bit quantisation produces
# equivalent results (same weight names, same layer structure).
BASE_MODEL = "mistral-community/pixtral-12b"
ADAPTER = "BenBarr/helpstral"

print("[1] Loading processor...")
processor = AutoProcessor.from_pretrained(BASE_MODEL)

print("[2] Loading base model (4-bit for T4 VRAM)...")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = LlavaForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    quantization_config=quantization_config,
    device_map="auto",
)

print("[3] Loading LoRA adapter...")
model = PeftModel.from_pretrained(model, ADAPTER)
model = model.merge_and_unload()
model.eval()

print(f"Model loaded on {next(model.parameters()).device}")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

[1] Loading processor...
[2] Loading base model (4-bit for T4 VRAM)...
Loading checkpoint shards: 100%|██████████| 5/5 [00:22<00:00,  4.51s/it]
[3] Loading LoRA adapter...
Model loaded on cuda:0
GPU memory: 13.42 GB


## Inference prompt — structured safety assessment

In [3]:
HELPSTRAL_PROMPT = """Analyze this drone camera frame. The escorted user is walking alone at night.

Output ONLY a valid JSON object with these exact keys:
- threat_level: integer 1-10
- status: SAFE, CAUTION, or DISTRESS
- people_count: integer
- user_moving: boolean
- proximity_alert: boolean
- observations: array of strings (2-4)
- pattern: string
- reasoning: string (2-3 sentences)
- action: CONTINUE_MONITORING, INCREASE_SCAN_RATE, ALERT_USER, ACTIVATE_SPOTLIGHT, or EMERGENCY_HOVER

Example: {"threat_level": 1, "status": "SAFE", "people_count": 1, "user_moving": true, "proximity_alert": false, "observations": ["well-lit street"], "pattern": "Normal", "reasoning": "...", "action": "CONTINUE_MONITORING"}"""

## Quick test

In [4]:
from PIL import Image
import json

test_img = Image.new("RGB", (320, 320), color=(40, 40, 60))

chat = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": HELPSTRAL_PROMPT}]}]
prompt = processor.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
inputs = processor(text=prompt, images=[test_img], return_tensors="pt")
inputs = {k: v.to(model.device) if hasattr(v, 'to') else v for k, v in inputs.items()}

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=400, do_sample=False)

text = processor.batch_decode(out, skip_special_tokens=True)[0]
json_part = text.split("[/INST]")[-1].strip()
print(json_part[:500])
print("\nInference OK!")

{"threat_level": 2, "status": "SAFE", "people_count": 1, "user_moving": true, "proximity_alert": false, "observations": ["dimly lit residential street", "single person walking at steady pace", "no other pedestrians visible"], "pattern": "Consistent safe \u2014 no changes detected", "reasoning": "The scene shows a residential area with moderate street lighting. One person is visible walking steadily along the pavement. No other individuals or vehicles in frame.", "action": "CONTINUE_MONITORING"}

Inference OK!


## Start API server

In [5]:
import base64, io, json, re, time
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "model": ADAPTER})

@app.route("/predict", methods=["POST"])
def predict():
    t0 = time.time()
    data = request.json
    image_b64 = data.get("image", "")
    if not image_b64:
        return jsonify({"error": "no image provided"}), 400

    img_bytes = base64.b64decode(image_b64)
    img = Image.open(io.BytesIO(img_bytes)).convert("RGB")

    chat = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": HELPSTRAL_PROMPT}]}]
    prompt = processor.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[img], return_tensors="pt")
    inputs = {k: v.to(model.device) if hasattr(v, 'to') else v for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=400, do_sample=False)

    text = processor.batch_decode(out, skip_special_tokens=True)[0]
    json_str = text.split("[/INST]")[-1].strip()
    match = re.search(r'\{.*\}', json_str, re.DOTALL)
    if match:
        try:
            result = json.loads(match.group())
        except json.JSONDecodeError:
            result = {"threat_level": 1, "status": "SAFE", "error": "parse_failed"}
    else:
        result = {"threat_level": 1, "status": "SAFE", "raw": json_str[:200]}

    result["inference_ms"] = int((time.time() - t0) * 1000)
    return jsonify(result)

print("Flask app ready")

Flask app ready


In [6]:
from pyngrok import ngrok

# ngrok.set_auth_token("YOUR_NGROK_TOKEN")
public_url = ngrok.connect(5000)
print("=" * 60)
print(f"HELPSTRAL ENDPOINT: {public_url}")
print(f"\nSet in .env: HELPSTRAL_ENDPOINT={public_url}")
print("=" * 60)
app.run(port=5000)

HELPSTRAL ENDPOINT: NgrokTunnel: "https://woefully-uncommon-bluebird.ngrok-free.dev" -> "http://localhost:5000"

Set in .env: HELPSTRAL_ENDPOINT=https://woefully-uncommon-bluebird.ngrok-free.dev
 * Serving Flask app '__main__'
 * Debug mode: off
INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [01/Mar/2026 14:12:33] "GET /health HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [01/Mar/2026 14:13:01] "POST /predict HTTP/1.1" 200 -
